In [1]:
import numpy as np
from scipy import optimize
from scipy.constants import mu_0, epsilon_0
from scipy import fftpack
from scipy import sparse
from scipy.special import factorial
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d, CubicSpline,splrep, BSpline
from scipy.sparse import csr_matrix, csc_matrix
import csv
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from IPython.display import display, Math, Latex, Markdown
from scipy.linalg import lu_factor, lu_solve
from scipy import signal
import ipywidgets as widgets
import empymod
import discretize
import  os
import glob
import json
import pandas as pd



In [2]:
from masa_utils import sci_latex
from masa_utils import InducedPolarizationSimulation
from masa_utils import Pelton_res_f, DDR_f, DDC_f
 

In [3]:
plt.style.use('00_video.mplstyle')


In [4]:
# --- helper to render LaTeX labels (robust method) ---
def math_label(tex, width="160px"):
    out = widgets.Output(layout=widgets.Layout(width=width))
    with out:
        display(Math(tex))
    return out
def sci_latex(v, prec=2):
    s = f"{v:.{prec}e}"          # e.g. '3.00e-03'
    mant, exp = s.split('e')
    exp = int(exp)

    if float(mant) == 0:
        return "0"

    if exp in [0, -1]:
        return f"{float(mant)*10**exp:.{prec}f}"
    else:
        return rf"{mant}\cdot 10^{{{exp}}}"


def fmt(v, prec=2, latex=False, wrap=False):
    # blank for None or empty string
    if v is None or v == "":
        return ""

    # if already string → return as-is
    if isinstance(v, str):
        return v

    try:
        s = sci_latex(v, prec=prec)

        if latex:
            if wrap:
                return rf"$${s}$$"   # display math
            else:
                return rf"${s}$"     # inline math
        else:
            return s

    except:
        return ""


In [5]:
rho0= 0.3
res_ref = 0.3
eta= 0.5
expc= 0.6

tau_rho = 1.0e-2

param_text = r"$\rho_0$" + f": {rho0: .2f}, " + r"$\eta$" +f": {eta: .2f}, "  
param_text += r"$\tau_\rho$"+ f": ${fmt(tau_rho, prec=1)}$, " +r"$C$" + f": {expc: .2f}"
mvec_true = np.array([np.log(rho0), eta, np.log(tau_rho), expc])

In [6]:
param_text

'$\\rho_0$:  0.30, $\\eta$:  0.50, $\\tau_\\rho$: $1.0\\cdot 10^{-2}$, $C$:  0.60'

Prepare Widget

In [7]:

freq_start_log = -1
freq_end_log = 4
freq_step_log = 0.1
nfreq= int((freq_end_log - freq_start_log) / freq_step_log) + 1
freq = np.logspace(freq_start_log, freq_end_log, nfreq)
IP_model = Pelton_res_f(freq=freq, chglim=0)
sim = InducedPolarizationSimulation(ip_model=IP_model, mode="sip")
freq_min, freq_max = freq.min(), freq.max()

In [8]:
log10rho0min, log10rho0max = -1, 0
rho0min, rho0max = 10**log10rho0min, 10**log10rho0max
etamin, etamax = 0, 0.5
log10taumin, log10taumax = -4, 1
taumin, taumax = 10**log10taumin, 10**log10taumax
cmin, cmax = 0.2, 1.0
rho0_default, eta_default, tau_default, c_default = 0.3, 0.0,  1e-3, 0.5


In [55]:
def draw_slider(ax, label, val, vmin, vmax, y, logscale=False, fmt_tau=False):
    # normalize value → [0,1]
    if logscale:
        t = (np.log10(val) - np.log10(vmin)) / (np.log10(vmax) - np.log10(vmin))
    else:
        t = (val - vmin) / (vmax - vmin)

    # background line
    ax.plot([0.2, 0.8], [y, y], color='0.7', lw=6, solid_capstyle='round')

    # knob
    x = 0.2 + 0.6 * t
    ax.plot(x, y, 'o', color='white', markeredgecolor='black', markersize=10)

    # label
    ax.text(0.05, y, label, va='center', fontsize=14)

    # current value
    if fmt_tau:
        ax.text(0.85, y, f"${fmt(val, prec=1)}$", va='center', fontsize=14)
    else:
        ax.text(0.85, y, f"{val:.2f}", va='center', fontsize=14)

    # ---- NEW: min/max labels ----
    if fmt_tau:
        vmin_str = f"${fmt(vmin, prec=1)}$"
        vmax_str = f"${fmt(vmax, prec=1)}$"
    else:
        vmin_str = f"{vmin:.2f}"
        vmax_str = f"{vmax:.2f}"

    ax.text(0.2, y-0.08, vmin_str, ha='center', va='top', color='0.3')
    ax.text(0.8, y-0.08, vmax_str, ha='center', va='top', color='0.3')

In [ ]:
#  Widget for the main function
def plot_pelton_widget(
         rho0, eta, tau, c, widget=True,
         eta_hline_threshold=0.10, axsip=None,axui=None):

    # Create the plot
    if widget:
        fig, ax = plt.subplots(3, 1)
        axsip = [ax[0], ax[1]]
        axui = ax[2]
    else:
        assert axsip is not None
        assert axui is not None
        axsip[0].clear()
        axsip[1].clear()
        axui.clear()

    m = np.r_[np.log(rho0), eta, np.log(tau), c]
    axsip = sim.plot_sip_model(m, ax=axsip, color='C0', label="")
    sim.ip_model.store_parameters(m)
    tau_rho, tau_psi, tau_con = sim.ip_model.tau_rho, sim.ip_model.tau_psi, sim.ip_model.tau_con
    axsip[0].set_xlim(left=freq_max, right=freq_min)
    if rho0*eta >= eta_hline_threshold:
        axsip[0].axhline(y=rho0, color='k', linestyle='--', label=r"$\rho_0$")
        axsip[0].axhline(y=rho0*(1-eta), color='k', linestyle='--', label=r"$\rho_{\infty}: \rho_0(1-\eta)$ ")
    axsip[1].set_xlim(left=freq_max, right=freq_min)
    axsip[1].axvline(x=1/2/np.pi/tau_con, color='k', linestyle='-.', label=r"$(2\pi\tau_\sigma)^{-1}$")   
    axsip[1].axvline(x=1/2/np.pi/tau_psi, color='k', linestyle='--', label=r"$(2\pi\tau_\phi)^{-1}$")
    axsip[1].axvline(x=1/2/np.pi/tau_rho, color='k', linestyle=':' , label=r"$(2\pi\tau_\rho)^{-1}$")
    for a in axsip:
        a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    axsip[0].set_ylim([0  ,1.1])
    axsip[0].set_xlabel("")

    axsip[1].set_ylim([1,-20])

    axui.axis("off")

    draw_slider(axui, r"$\rho_0\,(\Omega\,\mathrm{m})$", rho0, rho0min, rho0max, 0.75)
    draw_slider(axui, r"$\eta$", eta, etamin, etamax, 0.5)
    draw_slider(axui, r"$\tau_{\rho}\,(\mathrm{s})$", tau, taumin, taumax, 0.25, logscale=True, fmt_tau=True)
    draw_slider(axui, r"$C$", c, cmin, cmax, 0.0)

    axui.set_xlim(0, 1)
    axui.set_ylim(-0.1, 1)

    if widget:
        return fig, axsip, axui
    


In [ ]:

# --- sliders ---
rho0_slider = widgets.FloatLogSlider(
    base=10, min=log10rho0min, max=log10rho0max, step=0.01, value=rho0_default, description=""
)

eta_slider = widgets.FloatSlider(
    min=etamin, max=etamax, step=0.01, value=eta_default, description=""
)

tau_slider = widgets.FloatLogSlider(
    base=10, min=log10taumin, max=log10taumax, step=0.01, value=tau_default, description=""
)

c_slider = widgets.FloatSlider(
    min=cmin, max=cmax, step=0.01, value=c_default, description=""
)


In [43]:
# --- labeled rows ---
rho0_row = widgets.HBox([
    math_label(r"\rho_0\,(\Omega\,\mathrm{m})"),
    rho0_slider
])

eta_row = widgets.HBox([
    math_label(r"\eta"),
    eta_slider
])

tau_row = widgets.HBox([
    math_label(r"\tau_{\rho}\,(\mathrm{s})"),
    tau_slider
])

c_row = widgets.HBox([
    math_label(r"c"),
    c_slider
])


In [44]:
# --- controls layout ---
controls = widgets.VBox([rho0_row, eta_row, tau_row, c_row])
# --- connect UI to function (NO duplicate sliders) ---
out = widgets.interactive_output(
    plot_pelton_widget,
    {
        "rho0": rho0_slider,
        "eta": eta_slider,
        "tau": tau_slider,
        "c": c_slider,
    },
)

# --- display everything ---
display(controls, out)

Output()

In [14]:
def draw_pelton_on_axes(axsip, axui, rho0, eta, tau, c, eta_hline_threshold=0.10):
    # Redraw everything each frame (good for “consistency check”)
    axsip[0].clear()
    axsip[1].clear()
    axui.clear()

    # --- model ---
    m = np.r_[np.log(rho0), eta, np.log(tau), c]
    sim.plot_sip_model(m, ax=axsip, color='C0', label="")
    sim.ip_model.store_parameters(m)
    tau_rho, tau_psi, tau_con = sim.ip_model.tau_rho, sim.ip_model.tau_psi, sim.ip_model.tau_con

    # --- force consistent axes every frame ---
    axsip[0].set_xlim(left=freq_max, right=freq_min)
    axsip[1].set_xlim(left=freq_max, right=freq_min)
    axsip[0].set_ylim([0, 1.1])
    axsip[1].set_ylim([1, -15])
    axsip[0].set_xlabel("")

    # --- optional hlines ---
    if rho0 * eta >= eta_hline_threshold:
        axsip[0].axhline(y=rho0, color='k', linestyle='--', label=r"$\rho_0$")
        axsip[0].axhline(y=rho0*(1-eta), color='k', linestyle='--',
                         label=r"$\rho_{\infty}: \rho_0(1-\eta)$ ")

    # --- vlines on phase axis (your original choice) ---
    axsip[1].axvline(x=1/(2*np.pi*tau_con), color='k', linestyle='-.', label=r"$1/2/\pi/\tau_\sigma$")
    axsip[1].axvline(x=1/(2*np.pi*tau_psi), color='k', linestyle='--', label=r"$1/2/\pi/\tau_\psi$")
    axsip[1].axvline(x=1/(2*np.pi*tau_rho), color='k', linestyle=':',  label=r"$1/2/\pi/\tau_\rho$")

    # --- legends (outside, but kept visible by reserving space) ---
    for a in axsip:
        a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))

    # --- UI panel ---
    axui.axis("off")
    draw_slider(axui, r"$\rho_0\,(\Omega\,\mathrm{m})$", rho0, rho0min, rho0max, 0.75)
    draw_slider(axui, r"$\eta$", eta, etamin, etamax, 0.5)
    draw_slider(axui, r"$\tau_{\rho}\,(\mathrm{s})$", tau, taumin, taumax, 0.25, logscale=True, fmt=fmt)
    draw_slider(axui, r"$C$", c, cmin, cmax, 0.0)
    axui.set_xlim(0, 1)
    axui.set_ylim(-0.1, 1)


In [57]:
# -----------------------------
# 2) Define your base values and sweep ranges
# -----------------------------
# Use your defaults here:
rho0_base = 1.0
eta_base_1, eta_base_2  = 0, 0.5
tau_base  = tau_default
c_base    = c_default

# Choose sweep ranges (edit these!)
rho0_vals_1 = np.linspace(start=0.1, stop=1.0, num=121)
eta_vals  = np.linspace(0, 0.5, 121)
rho0_vals_2 = np.r_[np.linspace(start=1.0, stop=0.5, num=121)]
tau_vals  = np.r_[np.logspace(start=-3, stop=1.0,base=10, num=121)]
c_vals    = np.r_[np.linspace(0.5, 1.0, 76), 
                  np.linspace(1.0, 0.2,121), 
                  np.linspace(0.2, 0.5, 46)]

# Optional: keep your freq limits if you have them


# -----------------------------
# 3) Build a list of parameter states (ρ0 then η then τ then c)
# -----------------------------
states = [[], [], [], [], []]

# sweep rho0
for r in rho0_vals_1:
    states[0].append((r, eta_base_1, tau_base, c_base))
for r in rho0_vals_1[::-1]:  # reverse the list to go back to the start
    states[0].append((r, eta_base_1, tau_base, c_base))

# sweep eta
for e in eta_vals:
    states[1].append((rho0_base, e, tau_base, c_base))
for e in eta_vals[::-1]:  # reverse the list to go back to the star
    states[1].append((rho0_base, e, tau_base, c_base))

for r in rho0_vals_2:
    states[2].append((r, eta_base_2, tau_base, c_base))
for r in rho0_vals_2[::-1]:  # reverse the list to go back to the start
    states[2].append((r, eta_base_2, tau_base, c_base))

# sweep tau_rho
for t in tau_vals:
    states[3].append((rho0_base, eta_base_2, t, c_base))
for t in tau_vals[::-1]:  # reverse the list to go back to the start
    states[3].append((rho0_base, eta_base_2, t, c_base))

# sweep c
for cc in c_vals:
    states[4].append((rho0_base, eta_base_2, tau_base, cc))
len(states[0])

242

In [58]:
states

[[(np.float64(0.1), 0, 0.001, 0.5),
  (np.float64(0.10750000000000001), 0, 0.001, 0.5),
  (np.float64(0.115), 0, 0.001, 0.5),
  (np.float64(0.12250000000000001), 0, 0.001, 0.5),
  (np.float64(0.13), 0, 0.001, 0.5),
  (np.float64(0.1375), 0, 0.001, 0.5),
  (np.float64(0.14500000000000002), 0, 0.001, 0.5),
  (np.float64(0.15250000000000002), 0, 0.001, 0.5),
  (np.float64(0.16), 0, 0.001, 0.5),
  (np.float64(0.1675), 0, 0.001, 0.5),
  (np.float64(0.17500000000000002), 0, 0.001, 0.5),
  (np.float64(0.1825), 0, 0.001, 0.5),
  (np.float64(0.19), 0, 0.001, 0.5),
  (np.float64(0.1975), 0, 0.001, 0.5),
  (np.float64(0.20500000000000002), 0, 0.001, 0.5),
  (np.float64(0.21250000000000002), 0, 0.001, 0.5),
  (np.float64(0.22000000000000003), 0, 0.001, 0.5),
  (np.float64(0.2275), 0, 0.001, 0.5),
  (np.float64(0.23500000000000001), 0, 0.001, 0.5),
  (np.float64(0.24250000000000002), 0, 0.001, 0.5),
  (np.float64(0.25), 0, 0.001, 0.5),
  (np.float64(0.2575), 0, 0.001, 0.5),
  (np.float64(0.265), 0,

In [16]:
# -----------------------------
# settings
# # -----------------------------
out_mp4_list = ["../figures/37_sip_animarion_1.mp4",
           "../figures/37_sip_animarion_2.mp4",
           "../figures/37_sip_animarion_3.mp4",
           "../figures/37_sip_animarion_4.mp4",
           "../figures/37_sip_animarion_5.mp4"]


In [17]:
# -----------------------------
# video writer (no PNGs)
# -----------------------------
fps = 60
dpi = 150
writer = animation.FFMpegWriter(fps=fps, metadata={"artist": "matplotlib"})

for states_id, out_mp4 in enumerate(out_mp4_list):
    # fig, ax = plt.subplots(3, 1)
    # axsip = [ax[0], ax[1]]
    # axui = ax[2]

    # # KEY: reserve space so legends outside axes are still inside the figure canvas
    # fig.subplots_adjust(right=0.78)
    rho0, eta, tau, c = states[states_id][0]
    fig, axsip, axui = plot_pelton_widget(rho0, eta, tau, c)


    with writer.saving(fig, out_mp4, dpi=dpi):
        for i in range(1, len(states[states_id])):
            rho0, eta, tau, c = states[states_id][i]
            # draw_pelton_on_axes(axsip, axui, rho0, eta, tau, c)
            plot_pelton_widget(rho0, eta, tau, c, widget=False, axsip=axsip, axui=axui)
            fig.subplots_adjust(right=0.78)
            fig.canvas.draw()     # ensure the canvas is updated
            writer.grab_frame()   # grab this frame from *this same fig*

    plt.close(fig)
    print(f"Saved animation {states_id+1}: {out_mp4}")

C:\Users\81805\AppData\Local\Temp\ipykernel_9008\2314015753.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
C:\Users\81805\AppData\Local\Temp\ipykernel_9008\2314015753.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))


Saved animation 1: ../figures/37_sip_animarion_1.mp4


C:\Users\81805\AppData\Local\Temp\ipykernel_9008\2314015753.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))


Saved animation 2: ../figures/37_sip_animarion_2.mp4
Saved animation 3: ../figures/37_sip_animarion_3.mp4
Saved animation 4: ../figures/37_sip_animarion_4.mp4
Saved animation 5: ../figures/37_sip_animarion_5.mp4


Animation about challenges with Cole-Cole model 

In [45]:
file_path = "34_Pelton_TEM.json"
with open(file_path, "r") as f:
    data = json.load(f)
mpred_34 = np.array(data["models_rec_phd_avg"])
m_delta = np.array(data["V_pelton_min_avg"])
mvec_obs_34 = np.array(data["mvec_obs"])


In [52]:
#  Widget for the main function
def plot_pelton_challenge_widget(
         rho0, eta, tau, c, widget=True,
         eta_hline_threshold=0.10, axsip=None,axui=None):

    # Create the plot
    if widget:
        fig, ax = plt.subplots(3, 1)
        axsip = [ax[0], ax[1]]
        axui = ax[2]
    else:
        assert axsip is not None
        assert axui is not None
        axsip[0].clear()
        axsip[1].clear()
        axui.clear()

    m = np.r_[np.log(rho0), eta, np.log(tau), c]
    axsip = sim.plot_sip_model(m, ax=axsip, color='C0', label="Estimate")
    axsip = sim.plot_sip_model(mvec_obs_34, ax=axsip,  linestyle=":",color='C1', label="True")
    sim.ip_model.store_parameters(m)
    # tau_rho, tau_psi, tau_con = sim.ip_model.tau_rho, sim.ip_model.tau_psi, sim.ip_model.tau_con
    axsip[0].set_xlim(left=freq_max, right=freq_min)
    axsip[1].set_xlim(left=freq_max, right=freq_min)
    axsip[0].set_ylim([0.15, 0.30])
    axsip[1].set_ylim([0, -7])
    for a in axsip:
        a.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    # axsip[0].set_ylim([0  ,0.30])
    axsip[0].set_xlabel("")
    # axsip[1].set_ylim([1,-8])

    axui.axis("off")

    draw_slider(axui, r"$\rho_0\,(\Omega\,\mathrm{m})$", rho0, rho0min, rho0max, 0.75)
    draw_slider(axui, r"$\eta$", eta, etamin, etamax, 0.5)
    draw_slider(axui, r"$\tau_{\rho}\,(\mathrm{s})$", tau, taumin, taumax, 0.25, logscale=True, fmt_tau=True)
    draw_slider(axui, r"$C$", c, cmin, cmax, 0.0)

    axui.set_xlim(0, 1)
    axui.set_ylim(-0.1, 1)

    if widget:
        return fig, axsip, axui
    


In [53]:
states = []
num = 121

scale = np.linspace(start=-1.0, stop= 1.0, num=121)
scale = np.r_[scale, scale[::-1]]
for i in range(num*2):
    states_m = mpred_34 + scale[i] * m_delta
    states_m[0] = np.exp(states_m[0])
    states_m[2] = np.exp(states_m[2])
    states.append(states_m)


In [56]:
# -----------------------------
# video writer (no PNGs)
# -----------------------------
fps = 60
dpi = 150
writer = animation.FFMpegWriter(fps=fps, metadata={"artist": "matplotlib"})
out_mp4= "../figures/37_Pelton_challenge_animation.mp4"

rho0, eta, tau, c = states[0]
fig, axsip, axui = plot_pelton_widget(rho0, eta, tau, c)


with writer.saving(fig, out_mp4, dpi=dpi):
    for i in range(1, len(states)):
        rho0, eta, tau, c = states[i]
        # draw_pelton_on_axes(axsip, axui, rho0, eta, tau, c)
        plot_pelton_challenge_widget(rho0, eta, tau, c, widget=False, axsip=axsip, axui=axui)
        fig.subplots_adjust(right=0.78)
        fig.canvas.draw()     # ensure the canvas is updated
        writer.grab_frame()   # grab this frame from *this same fig*

plt.close(fig)
print(f"Saved animation: {out_mp4}")

Saved animation: ../figures/37_Pelton_challenge_animation.mp4
